In [ ]:
# -*- coding: utf-8 -*-
"""
Kurly 도시락·밥류(912003)
- 리스트에서 product_name 먼저 추출 후 ["도시락","김밥","컵밥","죽"] 포함 상품만 상세 진입
- 상품당 최대 500개 리뷰 수집(다음 버튼 클릭, iframe 대응, 무한 스크롤 방지)
- 저장 컬럼: category_name, product_name, brand, product_url, price, review_count, review_date, review_text
"""

import re, json, time
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from bs4 import BeautifulSoup

import undetected_chromedriver as uc
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# =========================== 설정 ===========================
CATEGORY_ID = "912003"
CATEGORY_NAME = "도시락·밥류"
PAGES_TO_CRAWL = 3
MAX_PRODUCTS = None                # None=모두, 정수면 상한
MAX_REVIEWS_PER_PRODUCT = 500
PAGINATION_TRIES_LIMIT = 1000
HEADLESS = False
WIN_SIZE = "1920,1080"
SHORT, MID, LONG = 3, 7, 12
DIGIT_ONLY = re.compile(r"^[0-9,]+$")
DATE_PAT = re.compile(r"\d{4}\.\d{1,2}\.\d{1,2}")
TARGET_KEYWORDS = ["도시락", "김밥", "컵밥", "죽"]

def is_target_product(name: str) -> bool:
    return bool(name) and any(k in name for k in TARGET_KEYWORDS)

# =========================== 드라이버 =========================
def make_driver() -> uc.Chrome:
    opts = uc.ChromeOptions()
    if HEADLESS: opts.add_argument("--headless=new")
    opts.add_argument("--disable-dev-shm-usage"); opts.add_argument("--no-sandbox")
    opts.add_argument("--lang=ko_KR"); opts.add_argument(f"--window-size={WIN_SIZE}")
    opts.add_argument("--start-maximized"); opts.add_argument("--disable-blink-features=AutomationControlled")
    service = Service(ChromeDriverManager().install())
    d = uc.Chrome(service=service, options=opts)
    d.set_page_load_timeout(60)
    return d

def wait_css(d, css, timeout=MID):  return WebDriverWait(d, timeout).until(EC.presence_of_element_located((By.CSS_SELECTOR, css)))
def wait_xpath(d, xp, timeout=MID): return WebDriverWait(d, timeout).until(EC.presence_of_element_located((By.XPATH, xp)))

def js_click(d, el):
    d.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
    time.sleep(0.15)
    d.execute_script("arguments[0].click();", el)

# =========================== 리스트 → (이름,링크) 수집 ====================
def list_page_url(page:int)->str:
    return f"https://www.kurly.com/categories/{CATEGORY_ID}?filters=&page={page}"

def _guess_name_from_card(anchor) -> str:
    # 카드 컨테이너 추정
    try:
        card = anchor.find_element(By.XPATH, "./ancestor::li[1] | ./ancestor::div[contains(@class,'grid') or contains(@class,'item')][1]")
    except Exception:
        card = None
    texts = []
    nodes = []
    if card:
        nodes += card.find_elements(By.XPATH, ".//strong|.//span|.//em|.//div")
    else:
        nodes += anchor.find_elements(By.XPATH, ".//strong|.//span|.//em|.//div")
    for n in nodes:
        try:
            t = n.text.strip()
            if not t: continue
            if "담기" in t: continue
            if "원" in t and DIGIT_ONLY.match(re.sub(r"[원,\s]","",t)):  # 가격 제거
                continue
            if len(t) > 100: continue
            texts.append(t)
        except: pass
    if texts:
        def score(s): return len(s) + (20 if re.search("[가-힣]", s) else 0) - (10 if "원" in s else 0)
        return max(texts, key=score)
    return (anchor.get_attribute("aria-label") or anchor.get_attribute("title") or anchor.text or "").strip()

def collect_filtered_links_from_list_page(d, page:int)->List[Tuple[str,str]]:
    d.get(list_page_url(page))
    wait_css(d, "div#container")
    # lazy-load 대비 적당 스크롤 (버튼 찾으면 더 이상 스크롤 안 함)
    for _ in range(10):
        d.execute_script("window.scrollBy(0, 1400);"); time.sleep(0.45)
    anchors = d.find_elements(By.CSS_SELECTOR, "a[href*='/goods/']")
    seen, results = set(), []
    for a in anchors:
        href = a.get_attribute("href") or ""
        if "/goods/" not in href or href in seen: continue
        name = _guess_name_from_card(a)
        if is_target_product(name):
            seen.add(href)
            results.append((name, href))
    return results

# =========================== 가격/리뷰수 ======================
def parse_json_ld(html:str)->Dict[str,Any]:
    soup = BeautifulSoup(html, "lxml"); out = {}
    for tag in soup.select('script[type="application/ld+json"]'):
        try: j = json.loads(tag.string)
        except: continue
        for o in (j if isinstance(j, list) else [j]):
            if isinstance(o, dict) and o.get("@type")=="Product":
                off = o.get("offers"); price = None
                if isinstance(off, dict): price = off.get("price")
                elif isinstance(off, list):
                    for oo in off:
                        if isinstance(oo, dict) and oo.get("price"): price = oo["price"]; break
                if price:
                    try: out["price"] = int(str(price).replace(",", ""))
                    except: pass
                ag = o.get("aggregateRating")
                if isinstance(ag, dict) and ag.get("reviewCount") is not None:
                    try: out["review_count"] = int(str(ag["reviewCount"]).replace(",", ""))
                    except: pass
                return out
    return out

def extract_price_dom(d)->Optional[int]:
    try:
        h1 = wait_xpath(d, "//h1", timeout=LONG)
        box = h1.find_element(By.XPATH, "./ancestor::section[1] | ./ancestor::div[2]")
        spans = box.find_elements(By.XPATH, ".//span[normalize-space()]")
        cands = []
        for s in spans:
            t = s.text.strip()
            if t and "%" not in t and DIGIT_ONLY.match(t):
                cands.append(int(t.replace(",", "")))
        if cands: return cands[0]
    except: pass
    return None

def extract_reviewcount_dom(d)->Optional[int]:
    try:
        node = d.find_element(By.XPATH, "//nav//*[contains(.,'후기')]/ancestor::li")
        txt = node.text.replace(",", ""); m = re.search(r"후기\s*\(?\s*(\d+)", txt)
        if m: return int(m.group(1))
    except: pass
    return None

# =========================== 제목 → 브랜드/상품명 ===============
def split_brand_from_title(title:str)->Tuple[Optional[str], str]:
    if not title: return None, ""
    m = re.match(r"\s*\[([^\]]+)\]\s*(.*)", title)
    if m:
        brand = m.group(1).strip(); name = m.group(2).strip() or title.strip()
        return brand, name
    return None, title.strip()

# =========================== 리뷰 섹션/페이지네이션 =============
def get_review_section(d):
    xps = [
        "//section[@aria-label='상품 후기' or contains(@aria-label,'상품 후기')]",
        "//*[@id='review' and (self::section or self::div)]",
        "//section[contains(@class,'review')]", "//div[@id='review']",
    ]
    for xp in xps:
        try:
            el = d.find_element(By.XPATH, xp)
            if el.is_displayed():
                d.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
                time.sleep(0.4); return el
        except: continue
    return None

def switch_into_review_iframe(d)->bool:
    try:
        frames = d.find_elements(By.CSS_SELECTOR, "#review iframe, section[aria-label] iframe, section[id*='review'] iframe, section[class*='review'] iframe")
        for fr in frames:
            if fr.is_displayed(): d.switch_to.frame(fr); return True
    except: pass
    return False

def leave_iframe(d):
    try: d.switch_to.default_content()
    except: pass

def _find_next_button_in(node):
    candidates = []
    for xp in [
        ".//button[not(@disabled)][.//div[normalize-space()='다음' or normalize-space()='다음 >']]",
        ".//button[not(@disabled)][.//span[normalize-space()='다음' or normalize-space()='다음 >']]",
        ".//div[./button][./button[2]]/button[2][not(@disabled)]",
        ".//button[not(@disabled)][contains(normalize-space(.), '>')]",
    ]:
        try:
            els = node.find_elements(By.XPATH, xp)
            vis = [e for e in els if e.is_displayed()]
            if vis: candidates.extend(vis)
        except: continue
    filtered = []
    for b in candidates:
        try:
            t = (b.text or "").strip()
            if "이전" not in t: filtered.append(b)
        except: pass
    if not filtered: return None
    if len(filtered)==1: return filtered[0]
    pri = [b for b in filtered if "다음" in (b.text or "")]
    return pri[0] if len(pri)==1 else filtered[-1]

def click_review_next(d)->bool:
    sec = get_review_section(d)
    if sec:
        btn = _find_next_button_in(sec)
        if btn: js_click(d, btn); return True
    entered = switch_into_review_iframe(d)
    try:
        if entered:
            for xp in [
                "//button[not(@disabled)][.//*[normalize-space()='다음' or normalize-space()='다음 >']]", 
                "//div[./button][./button[2]]/button[2][not(@disabled)]",
                "//button[not(@disabled)][contains(normalize-space(.), '>') and not(contains(.,'이전'))]"
            ]:
                try:
                    els = d.find_elements(By.XPATH, xp)
                    vis = [e for e in els if e.is_displayed()]
                    if vis: js_click(d, vis[-1] if len(vis)>1 else vis[0]); return True
                except: continue
    finally:
        if entered: leave_iframe(d)
    try:
        btn = d.find_element(By.XPATH, "//button[not(@disabled)][.//*[contains(.,'다음') or contains(.,'>')]][last()]")
        js_click(d, btn); return True
    except: return False

def find_first_review_card(d):
    for xp in ["//section[@aria-label='상품 후기']//article[1]", "//*[@id='review']//article[1]", "//article[1]"]:
        try:
            el = d.find_element(By.XPATH, xp)
            if el.is_displayed(): return el
        except: continue
    return None

def parse_reviews_now(d, product_url:str)->List[Dict[str,Any]]:
    soup = BeautifulSoup(d.page_source, "lxml")
    root = soup.select_one("section[aria-label='상품 후기']") or soup.select_one("#review") or soup.select_one("section[class*='review']") or soup
    cards = root.select("article") or root.select("li")
    rows=[]
    for tag in cards:
        ps = [p.get_text(" ", strip=True) for p in tag.select("p") if p.get_text(strip=True)]
        body = max(ps, key=len) if ps else tag.get_text(" ", strip=True)
        if not body or len(body)<10: continue
        m = DATE_PAT.search(tag.get_text(" ", strip=True))
        date_str = m.group(0) if m else None
        rows.append({"product_url":product_url, "review_date":date_str, "review_text":body})
        if len(rows) >= MAX_REVIEWS_PER_PRODUCT: break
    return rows

def paginate_and_collect_reviews(d, product_url:str, max_reviews:int=MAX_REVIEWS_PER_PRODUCT)->List[Dict[str,Any]]:
    for xp in [
        "//nav//li[.//span[contains(.,'후기')]]",
        "//nav//*[self::a or self::button][contains(.,'후기')]",
        "//button[contains(.,'후기')]", "//a[contains(.,'후기')]",
    ]:
        try:
            el = d.find_element(By.XPATH, xp); js_click(d, el); time.sleep(0.6); break
        except: continue

    rows = parse_reviews_now(d, product_url)
    seen = set((r["review_date"], r["review_text"]) for r in rows)

    tries = 0
    while len(rows) < max_reviews and tries < PAGINATION_TRIES_LIMIT:
        base = find_first_review_card(d)
        btn = None; sec = get_review_section(d)
        if sec: btn = _find_next_button_in(sec)
        if not btn:
            for _ in range(6):
                sec = get_review_section(d)
                btn = _find_next_button_in(sec) if sec else None
                if btn: break
                d.execute_script("window.scrollBy(0, 650);"); time.sleep(0.3)
        if not btn and not click_review_next(d): break
        if btn: js_click(d, btn)

        changed = False
        try:
            if base: WebDriverWait(d, LONG).until(EC.staleness_of(base)); changed = True
            else: time.sleep(1.0); changed = True
        except TimeoutException: pass

        new_rows = parse_reviews_now(d, product_url)
        added = 0
        for r in new_rows:
            key = (r["review_date"], r["review_text"])
            if key not in seen:
                rows.append(r); seen.add(key); added += 1
                if len(rows) >= max_reviews: break
        tries += 1
        if added == 0 and not changed: break
    return rows[:max_reviews] if max_reviews else rows

# =========================== 상세 크롤 =========================
def crawl_product(d, url:str)->List[Dict[str,Any]]:
    d.get(url); time.sleep(1.0)
    try: title = wait_xpath(d, "//h1", timeout=LONG).text.strip()
    except Exception:
        try: title = wait_css(d, "h1, h2", timeout=MID).text.strip()
        except: title = ""
    brand, product_name = split_brand_from_title(title)

    j = parse_json_ld(d.page_source)
    price = j.get("price") or extract_price_dom(d)
    review_count = j.get("review_count") or extract_reviewcount_dom(d)

    reviews = paginate_and_collect_reviews(d, url, MAX_REVIEWS_PER_PRODUCT)
    if not reviews: reviews = [{"product_url":url, "review_date":None, "review_text":None}]

    rows=[]
    for r in reviews:
        rows.append({
            "category_name": CATEGORY_NAME,
            "product_name": product_name,
            "brand": brand,
            "product_url": url,
            "price": price,
            "review_count": review_count,
            "review_date": r["review_date"],
            "review_text": r["review_text"],
        })
    return rows

# =========================== 메인 =============================
def main():
    d = make_driver()
    all_rows: List[Dict[str,Any]] = []
    try:
        # 1) 리스트에서 (이름,링크) 수집 후 키워드로 선필터
        pairs, seen = [], set()
        for p in range(1, PAGES_TO_CRAWL+1):
            pairs.extend(collect_filtered_links_from_list_page(d, p))
        if MAX_PRODUCTS: pairs = pairs[:MAX_PRODUCTS]
        print(f"[INFO] 필터 통과 상품 {len(pairs)}개")

        # 2) 필터링된 URL만 상세 진입
        for i, (name, url) in enumerate(pairs, 1):
            print(f"[{i}/{len(pairs)}] {name} -> {url}")
            try:
                rows = crawl_product(d, url)
                all_rows.extend(rows)
            except Exception as e:
                print(f"  └─ 에러: {e}")
    finally:
        d.quit()

    out = f"kurly_meal_rice_reviews_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    cols = ["category_name","product_name","brand","product_url","price","review_count","review_date","review_text"]
    pd.DataFrame(all_rows, columns=cols).to_csv(out, index=False, encoding="utf-8-sig")
    print(f"\n[DONE] 저장: {out} (행 {len(all_rows)})")

if __name__ == "__main__":
    main()


In [5]:
# naver_smartstore_reviews_merged.py
# -*- coding: utf-8 -*-

"""
Naver Shopping '정기구독 도시락' 예시
- 검색결과 다중 페이지 수집 (스마트스토어만)
- 광고(파워링크/스폰서) 제외
- 상세 JSON-LD 우선: price / ratingValue / reviewCount
- 리뷰 탭 페이지네이션: >= N개 수집
- 최종 저장 전, 상품/리뷰 중복 제거

출력 컬럼:
list_rank, list_page, list_name, list_url, product_id, list_price, list_review_count,
product_url, product_title, brand, price, rating_value, review_count_total,
review_date, review_text
"""

import re, json, time, random
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from bs4 import BeautifulSoup

# ── Selenium / Chrome
import undetected_chromedriver as uc
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from 

# ── (선택) urllib에서 같은 UA 쓰고 싶을 때 사용
from urllib.request import FancyURLopener

# =========================== 설정 ===========================
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

SEARCH_URL = (
    "https://search.shopping.naver.com/search/all?"
    "adQuery=%EC%A0%95%EA%B8%B0%EA%B5%AC%EB%8F%85%20%EB%8F%84%EC%8B%9C%EB%9D%BD"
    "&frm=NVSCTAB&origQuery=%EC%A0%95%EA%B8%B0%EA%B5%AC%EB%8F%85%20%EB%8F%84%EC%8B%9C%EB%9D%BD"
    "&pagingIndex=1&pagingSize=40&productSet=total"
    "&query=%EC%A0%95%EA%B8%B0%EA%B5%AC%EB%8F%85%20%EB%8F%84%EC%8B%9C%EB%9D%BD"
    "&sort=rel&timestamp=&viewType=list"
)

PAGES_TO_CRAWL = 3                 # 검색결과 페이지 수
MAX_PRODUCTS_PER_PAGE = None       # None=모두, 정수면 상한
MAX_REVIEWS_PER_PRODUCT = 200      # 상품당 리뷰 최대 수집(50 이상 확보 권장)
HEADLESS = False
WIN_SIZE = "1920,1080"
SHORT, MID, LONG = 3, 7, 12

DATE_PAT = re.compile(r"\d{4}\.\d{1,2}\.\d{1,2}")
DIGIT_ONLY = re.compile(r"^[0-9,]+$")
TARGET_DOMAIN = "smartstore.naver.com"   # 스마트스토어만 상세 진입

# =========================== urllib UA (선택) =================
class AppURLopener(FancyURLopener):
    # 네가 준 코드와 동일한 방식으로 UA 지정
    version = USER_AGENT

http = AppURLopener()  # 사용 예: http.open(url).read()

# =========================== Selenium 드라이버 =================
def make_driver() -> uc.Chrome:
    opts = uc.ChromeOptions()
    if HEADLESS: opts.add_argument("--headless=new")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--lang=ko_KR")
    opts.add_argument(f"--window-size={WIN_SIZE}")
    opts.add_argument("--start-maximized")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    # ✅ UA 지정 (네가 원하는 부분)
    opts.add_argument(f"--user-agent={USER_AGENT}")
    opts.add_argument()

    service = Service(ChromeDriverManager().install())
    d = uc.Chrome(service=service, options=opts)
    d.set_page_load_timeout(60)
    return d

def wait_css(d, css, timeout=MID):  return WebDriverWait(d, timeout).until(EC.presence_of_element_located((By.CSS_SELECTOR, css)))
def wait_xpath(d, xp, timeout=MID): return WebDriverWait(d, timeout).until(EC.presence_of_element_located((By.XPATH, xp)))

def js_click(d, el):
    d.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
    time.sleep(0.15)
    d.execute_script("arguments[0].click();", el)

# =========================== 유틸 ===========================
def set_url_page(url: str, page:int) -> str:
    u = urlparse(url)
    q = parse_qs(u.query)
    q["pagingIndex"] = [str(page)]
    new_q = urlencode({k: v[0] if isinstance(v, list) else v for k, v in q.items()})
    return urlunparse((u.scheme, u.netloc, u.path, u.params, new_q, u.fragment))

def clean_price_txt(txt:str) -> Optional[int]:
    if not txt: return None
    txt = txt.replace(",", "")
    m = re.search(r"(\d{3,})\s*원?", txt)
    if m:
        try: return int(m.group(1))
        except: return None
    return None

def parse_json_ld(html:str) -> Dict[str, Any]:
    soup = BeautifulSoup(html, "lxml"); out = {}
    for tag in soup.select('script[type="application/ld+json"]'):
        if not tag.string: continue
        try: j = json.loads(tag.string)
        except: continue
        objs = j if isinstance(j, list) else [j]
        for o in objs:
            if not isinstance(o, dict): continue
            if o.get("@type") in ("Product", "AggregateOffer", "Offer"):
                # 가격
                price = None
                off = o.get("offers")
                if isinstance(off, dict): price = off.get("price")
                elif isinstance(off, list):
                    for oo in off:
                        if isinstance(oo, dict) and oo.get("price"):
                            price = oo["price"]; break
                if price:
                    try: out["price"] = int(str(price).replace(",", ""))
                    except: pass
                # 평점/리뷰수
                ag = o.get("aggregateRating")
                if isinstance(ag, dict):
                    if "ratingValue" in ag:
                        try: out["rating_value"] = float(str(ag["ratingValue"]))
                        except: pass
                    if "reviewCount" in ag:
                        try: out["review_count"] = int(str(ag["reviewCount"]).replace(",", ""))
                        except: pass
    return out

def split_brand_from_title(title:str)->Tuple[Optional[str], str]:
    if not title: return None, ""
    m = re.match(r"\s*\[([^\]]+)\]\s*(.*)", title)
    if m:
        return m.group(1).strip(), (m.group(2) or title).strip()
    return None, title.strip()

# =========================== 광고/중복 필터 ====================
AD_CLASS_KEYS = ("adProduct", "power_link", "powerlink", "ad_area", "ad_wrap")
AD_TEXT_KEYS = ("광고", "파워링크", "스폰서")

def _card_container(el):
    try:
        return el.find_element(By.XPATH, "./ancestor::div[contains(@class,'product') or contains(@class,'basicList') or contains(@class,'adProduct') or contains(@class,'product_item') or contains(@class,'_item')][1]")
    except:
        try: return el.find_element(By.XPATH, "./ancestor::li[1]")
        except: return None

def is_ad_element(anchor) -> bool:
    """리스트 카드가 광고인지 휴리스틱으로 판정"""
    # (1) ad* 클래스 조상
    try:
        anc = anchor.find_element(By.XPATH, "./ancestor::*[contains(@class,'adProduct') or contains(@class,'power_link') or contains(@data-shp-area,'ad') or contains(@data-shp-areatype,'ad')][1]")
        if anc: return True
    except: pass
    # (2) 카드 내부 텍스트에 광고 키워드
    card = _card_container(anchor)
    if card:
        try:
            t = card.text
            if any(k in t for k in AD_TEXT_KEYS):
                return True
        except: pass
    return False

def extract_pid(href:str) -> str:
    m = re.search(r"/products/(\d+)", href)
    return m.group(1) if m else href

# =========================== 검색결과(리스트) =================
LIST_CARD_XPS = [
    # 광고/일반 레이아웃이 수시로 바뀌어도 '스마트스토어 제품 링크' 조건만으로 anchor 추출
    "//a[contains(@href,'smartstore.naver.com') and contains(@href,'/products/')]",
]

def _guess_list_name(card, anchor) -> str:
    cands = []
    nodes = []
    try: nodes += card.find_elements(By.XPATH, ".//strong|.//h3|.//span|.//div")
    except: nodes = [anchor]
    for n in nodes:
        try:
            t = n.text.strip()
            if not t: continue
            if "담기" in t or "쿠폰" in t: continue
            if "원" in t and DIGIT_ONLY.match(re.sub(r"[원,\s]", "", t)):  # 가격 텍스트 제외
                continue
            if len(t) > 120: continue
            cands.append(t)
        except: pass
    return max(cands, key=len) if cands else (anchor.get_attribute("title") or anchor.text or "").strip()

def parse_list_page(d, page:int, visited_ids:set) -> List[Dict[str, Any]]:
    url = set_url_page(SEARCH_URL, page)
    d.get(url)
    wait_css(d, "body")
    # lazy-load 대응 스크롤
    for _ in range(12):
        d.execute_script("window.scrollBy(0, 1600);"); time.sleep(0.35)
    time.sleep(0.5)

    anchors = []
    for xp in LIST_CARD_XPS:
        try:
            anchors = d.find_elements(By.XPATH, xp)
            if anchors: break
        except: continue

    rows = []
    rank = 0
    for a in anchors:
        href = a.get_attribute("href") or ""
        if TARGET_DOMAIN not in href or "/products/" not in href:
            continue
        # ✅ 광고 제외
        if is_ad_element(a):
            continue
        pid = extract_pid(href)
        # ✅ 전역 중복 제거
        if pid in visited_ids:
            continue
        visited_ids.add(pid)

        card = _card_container(a)
        name = _guess_list_name(card, a)

        # 리스트에서 가격/리뷰수 보이면 잡기(없으면 상세에서)
        list_price, list_reviews = None, None
        if card:
            try:
                ptxt = card.text
                list_price = clean_price_txt(ptxt)
                m2 = re.search(r"리뷰\s*([0-9,]+)", ptxt)
                if m2: list_reviews = int(m2.group(1).replace(",", ""))
            except: pass

        rank += 1
        rows.append({
            "list_rank": rank,
            "list_page": page,
            "list_name": name,
            "list_url": href,
            "product_id": pid,
            "list_price": list_price,
            "list_review_count": list_reviews,
        })
        if MAX_PRODUCTS_PER_PAGE and len(rows) >= MAX_PRODUCTS_PER_PAGE:
            break
    return rows

# =========================== 상세/리뷰 ========================
def extract_title_dom(d)->str:
    for xp in ["//h1", "//h2", "//header//h1"]:
        try:
            t = d.find_element(By.XPATH, xp).text.strip()
            if t: return t
        except: continue
    return ""

def extract_price_dom(d)->Optional[int]:
    try:
        box = d.find_element(By.XPATH, "//h1/ancestor::section[1] | //h1/ancestor::div[2]")
        spans = box.find_elements(By.XPATH, ".//span[normalize-space()]")
        for s in spans:
            t = s.text.strip()
            if t and DIGIT_ONLY.match(t.replace(",", "")):
                return int(t.replace(",", ""))
    except: pass
    try:
        return clean_price_txt(d.find_element(By.TAG_NAME, "body").text)
    except: return None

def extract_reviewtab_count_dom(d)->Optional[int]:
    try:
        node = d.find_element(By.XPATH, "//nav//*[contains(.,'리뷰') or contains(.,'후기')]/ancestor::li")
        txt = node.text.replace(",", "")
        m = re.search(r"(리뷰|후기)\s*\(?\s*(\d+)", txt)
        if m: return int(m.group(2))
    except: pass
    return None

def goto_review_tab(d) -> bool:
    for xp in [
        "//nav//*[self::a or self::button][contains(.,'리뷰') or contains(.,'후기')]",
        "//a[contains(.,'리뷰') or contains(.,'후기')]",
        "//button[contains(.,'리뷰') or contains(.,'후기')]",
        "//*[@id='REVIEW' or @id='review']",
    ]:
        try:
            el = d.find_element(By.XPATH, xp)
            js_click(d, el); time.sleep(0.8)
            return True
        except: continue
    return False

def get_review_root(d):
    for xp in [
        "//section[@id='REVIEW' or contains(@aria-label,'리뷰') or contains(@aria-label,'상품 후기')]",
        "//*[@id='REVIEW' or @id='review']",
        "//section[contains(@class,'review')]",
    ]:
        try:
            el = d.find_element(By.XPATH, xp)
            if el.is_displayed():
                d.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
                return el
        except: continue
    return None

def parse_reviews_now(d, product_url:str)->List[Dict[str,Any]]:
    soup = BeautifulSoup(d.page_source, "lxml")
    root = (soup.select_one("#REVIEW") or 
            soup.select_one("section[aria-label*='리뷰'], section[aria-label*='후기']") or
            soup.select_one("section[class*='review']") or soup)
    cards = root.select("article") or root.select("ul li")
    out = []
    for tag in cards:
        txts = [p.get_text(" ", strip=True) for p in tag.select("p") if p.get_text(strip=True)]
        body = max(txts, key=len) if txts else tag.get_text(" ", strip=True)
        body = (body or "").strip()
        if not body or len(body) < 5: continue
        m = DATE_PAT.search(tag.get_text(" ", strip=True))
        date_str = m.group(0) if m else None
        out.append({"product_url": product_url, "review_date": date_str, "review_text": body})
    return out

def click_review_page(d, page_num:int) -> bool:
    try:
        nav = d.find_elements(By.XPATH, "//a[@role='menuitem' and normalize-space()]")
        for a in nav:
            if a.text.strip() == str(page_num) and a.get_attribute("aria-current") != "true":
                js_click(d, a); return True
        nxt = [a for a in nav if "다음" in a.text]
        if nxt:
            js_click(d, nxt[-1]); return True
    except: pass
    try:
        btn = d.find_element(By.XPATH, "//button[not(@disabled)][.//*[contains(.,'다음') or contains(.,'>')]]")
        js_click(d, btn); return True
    except: pass
    return False

def paginate_and_collect_reviews(d, product_url:str, need:int=MAX_REVIEWS_PER_PRODUCT)->List[Dict[str,Any]]:
    goto_review_tab(d)
    time.sleep(0.7)
    root = get_review_root(d)
    if root is None:
        for _ in range(6):
            d.execute_script("window.scrollBy(0, 900);"); time.sleep(0.3)
            root = get_review_root(d)
            if root: break

    rows = parse_reviews_now(d, product_url)
    seen = set((r["review_date"], r["review_text"]) for r in rows)
    page_idx = 1
    tries = 0
    while len(rows) < need and tries < 400:
        moved = click_review_page(d, page_idx + 1)
        if not moved:
            break
        try:
            WebDriverWait(d, LONG).until(EC.staleness_of(d.find_elements(By.XPATH, "//article|//ul/li")[0]))
        except Exception:
            time.sleep(0.9)
        time.sleep(0.6)
        new_rows = parse_reviews_now(d, product_url)
        added = 0
        for r in new_rows:
            key = (r["review_date"], r["review_text"])
            if key not in seen:
                rows.append(r); seen.add(key); added += 1
                if len(rows) >= need: break
        page_idx += 1
        tries += 1
        if added == 0 and not moved:
            break
    return rows[:need]

def crawl_product_detail(d, url:str) -> Dict[str, Any]:
    d.get(url); time.sleep(1.0)

    title = extract_title_dom(d)
    brand, prod_name = split_brand_from_title(title)
    j = parse_json_ld(d.page_source)
    price = j.get("price") or extract_price_dom(d)
    rating_value = j.get("rating_value")
    review_count_total = j.get("review_count") or extract_reviewtab_count_dom(d)

    reviews = paginate_and_collect_reviews(d, url, MAX_REVIEWS_PER_PRODUCT)
    if not reviews:
        reviews = [{"product_url": url, "review_date": None, "review_text": None}]

    rows = []
    for r in reviews:
        rows.append({
            "product_url": url,
            "product_title": prod_name or title,
            "brand": brand,
            "price": price,
            "rating_value": rating_value,
            "review_count_total": review_count_total,
            "review_date": r["review_date"],
            "review_text": r["review_text"],
        })
    return {
        "meta": {"product_url": url, "title": prod_name or title, "brand": brand,
                 "price": price, "rating_value": rating_value, "review_count_total": review_count_total},
        "rows": rows,
    }

# =========================== 메인 =============================
def main():
    d = make_driver()
    all_rows: List[Dict[str,Any]] = []
    visited_ids: set = set()   # ✅ 페이지 간 중복 제거용

    try:
        # 1) 검색결과 페이지 순회
        list_items: List[Dict[str,Any]] = []
        for p in range(1, PAGES_TO_CRAWL+1):
            items = parse_list_page(d, p, visited_ids)
            print(f"[LIST] page {p}: {len(items)} items (ads excluded)")
            list_items.extend(items)

        # 2) 스마트스토어 상세 진입 & 리뷰 수집
        for i, it in enumerate(list_items, 1):
            url = it["list_url"]
            print(f"[{i}/{len(list_items)}] {it['list_name']} -> {url}")
            try:
                res = crawl_product_detail(d, url)
                for row in res["rows"]:
                    row.update({
                        "list_rank": it["list_rank"],
                        "list_page": it["list_page"],
                        "list_name": it["list_name"],
                        "list_url": it["list_url"],
                        "product_id": it["product_id"],
                        "list_price": it["list_price"],
                        "list_review_count": it["list_review_count"],
                    })
                all_rows.extend(res["rows"])
            except Exception as e:
                print(f"  └─ 에러: {e}")

    finally:
        d.quit()

    # 3) 저장 (✅ 최종 중복 제거)
    ts = datetime.now().strftime("%Y%m%d_%H%M")
    out = f"naver_shopping_smartstore_reviews_{ts}.csv"

    df = pd.DataFrame(all_rows)
    # 리뷰 중복(같은 상품_url+날짜+내용) 제거
    if not df.empty:
        df.drop_duplicates(
            subset=["product_url", "review_date", "review_text"],
            inplace=True
        )
        # 혹시 동일 리뷰텍스트만 있는 경우(날짜 누락)도 한 번 더 정리
        df.drop_duplicates(
            subset=["product_url", "review_text"], inplace=True
        )

    cols_order = [
        "list_rank","list_page","list_name","list_url","product_id","list_price","list_review_count",
        "product_url","product_title","brand","price","rating_value","review_count_total",
        "review_date","review_text"
    ]
    df = df.reindex(columns=cols_order)

    df.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"\n[DONE] 저장: {out} (행 {len(df)})")

if __name__ == "__main__":
    main()


SyntaxError: invalid syntax (2572739569.py, line 33)

In [ ]:
# %% [markdown]
# 아이템스카우트에 직접 로그인한 뒤, 세션 쿠키를 저장합니다.
# 저장된 쿠키는 같은 기기/브라우저 드라이버 버전에서 일정 기간 재사용할 수 있습니다.

# %%
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import pickle, time, os

COOKIE_PATH = "itemscout_cookies.pkl"

opts = Options()
opts.add_argument("--start-maximized")
opts.add_argument("window-size=1920x1080")
opts.add_argument("lang=ko_KR")
# 필요시 사용자 에이전트 명시
# opts.add_argument("--user-agent=Mozilla/5.0")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

try:
    # 로그인 페이지로 이동 (또는 메인 -> 로그인)
    driver.get("https://itemscout.io/login")
    print("수동으로 로그인/인증을 완료하세요. (일회성) 60초 대기...")
    time.sleep(60)  # 필요시 늘리세요

    # 로그인 후 메인으로 한 번 이동해 세션이 안정적으로 잡히도록
    driver.get("https://itemscout.io/")
    time.sleep(3)

    # 쿠키 저장
    cookies = driver.get_cookies()
    with open(COOKIE_PATH, "wb") as f:
        pickle.dump(cookies, f)
    print(f"쿠키 저장 완료 → {os.path.abspath(COOKIE_PATH)}  (쿠키 개수: {len(cookies)})")

finally:
    driver.quit()


In [7]:
# %% [markdown]
# 저장한 쿠키를 로드해 로그인 상태로 키워드 페이지를 열고,
# 화면에 보이는 상품 카드 정보를 수집해 CSV로 저장합니다.

# %%
import re, time, pickle, os, pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

COOKIE_PATH = "itemscout_cookies.pkl"
TARGET_URL = "https://itemscout.io/keyword/244728814"
OUT_CSV = "itemscout_keyword_244728814.csv"

opts = Options()
opts.add_argument("--start-maximized")
opts.add_argument("window-size=1920x1080")
opts.add_argument("lang=ko_KR")
# opts.add_argument("--user-agent=Mozilla/5.0")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

def load_cookies_to_driver(driver, cookie_path, domain_hint=".itemscout.io"):
    if not os.path.exists(cookie_path):
        raise FileNotFoundError("쿠키 파일이 없습니다. 먼저 '쿠키 저장' 셀을 실행하세요.")
    # 도메인 컨텍스트 필요
    driver.get("https://itemscout.io/")
    time.sleep(2)
    with open(cookie_path, "rb") as f:
        cookies = pickle.load(f)
    # 도메인 미스매치/expiry 타입 보정
    for ck in cookies:
        c = ck.copy()
        c.pop("sameSite", None)   # 일부 드라이버에서 sameSite 값 이슈
        # 도메인이 다른 쿠키는 아이템스카우트 도메인으로 보정 시도
        if "domain" in c and "itemscout.io" not in c["domain"]:
            c["domain"] = domain_hint
        # expiry가 float/str이면 int로
        if "expiry" in c and not isinstance(c["expiry"], int):
            try: c["expiry"] = int(c["expiry"])
            except: c.pop("expiry", None)
        try:
            driver.add_cookie(c)
        except Exception:
            pass
    driver.refresh()
    time.sleep(2)

def is_ad(card) -> bool:
    """카드 내 '광고' 배지가 있으면 True"""
    try:
        if any(el.is_displayed() for el in card.find_elements(By.XPATH, ".//*[contains(text(),'광고')]")):
            return True
    except: 
        pass
    # 텍스트 전체 검사
    try:
        if "광고" in card.text:
            return True
    except:
        pass
    return False

def get_first_text(el, xpaths):
    for xp in xpaths:
        try:
            t = el.find_element(By.XPATH, xp).text.strip()
            if t: return t
        except: 
            continue
    return ""

def extract_with_regex(text, patterns, cast=str):
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            val = m.group(1).replace(",", "")
            try:
                return cast(val)
            except:
                return val
    return None

try:
    # 1) 쿠키 로드
    load_cookies_to_driver(driver, COOKIE_PATH)

    # 2) 타깃 페이지 열기
    driver.get(TARGET_URL)
    # 주요 리스트 영역 도착 대기 (제일 안정적인 건 카드 li 존재 여부)
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.XPATH, "//main//li"))
    )
    time.sleep(1)

    # 3) 카드 요소 수집
    # 아이템스카우트는 Tailwind/React 기반이라 클래스명이 자주 바뀜 → 텍스트 휴리스틱 사용
    cards = driver.find_elements(By.XPATH, "//main//li[.//text()[contains(.,'원')] or .//*[contains(text(),'리뷰')]]")
    rows = []
    rank = 0
    seen = set()  # (제목, 가격) 기준 중복 제거

    for card in cards:
        if not card.is_displayed():
            continue
        # 광고 제외
        if is_ad(card):
            continue

        # 텍스트 블록
        full = card.text

        # 제목: 카드 상단의 굵은 텍스트 혹은 첫 줄
        title = get_first_text(card, [
            ".//div[contains(@class,'font-bold')][1]",
            ".//p[contains(@class,'font-bold')][1]",
            ".//div[1]"
        ])
        if not title:
            # 첫 줄
            title = full.splitlines()[0].strip() if full.splitlines() else ""

        # 가격
        price = extract_with_regex(full, [r"([0-9,]+)\s*원"], int)

        # 리뷰수 / 평점 / 등록일
        review_cnt = extract_with_regex(full, [r"리뷰\s*([0-9,]+)"], int)
        rating = extract_with_regex(full, [r"평점\s*([0-9.]+)"], float)
        reg_date = extract_with_regex(full, [r"등록일\s*([0-9]{4}\.[0-9]{2}\.[0-9]{2})"], str)

        # 예상판매량(7일)
        expected7 = extract_with_regex(full, [r"예상판매량\(?7일\)?\s*([0-9,]+)"], int)

        # 찜/공감 등 수치(있으면)
        zzim = extract_with_regex(full, [r"찜\s*([0-9,]+)"], int)

        # 중복 방지
        dup_key = (title, price)
        if dup_key in seen:
            continue
        seen.add(dup_key)

        rank += 1
        rows.append({
            "rank": rank,
            "title": title,
            "price": price,
            "review_count": review_cnt,
            "rating": rating,
            "reg_date": reg_date,
            "expected_sales_7d": expected7,
            "zzim": zzim,
        })

    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"완료: {OUT_CSV} (행 {len(df)})")

finally:
    driver.quit()


UnexpectedAlertPresentException: Alert Text: 확장 프로그램 미설치로 데이터를 불러올 수 없습니다.
확장 프로그램 설치 버튼을 누르고 다시 시도해 주세요.
Message: unexpected alert open: {Alert text : 확장 프로그램 미설치로 데이터를 불러올 수 없습니다.
확장 프로그램 설치 버튼을 누르고 다시 시도해 주세요.}
  (Session info: chrome=140.0.7339.128)
Stacktrace:
	GetHandleVerifier [0x0xd1c333+65459]
	GetHandleVerifier [0x0xd1c374+65524]
	(No symbol) [0x0xb3d973]
	(No symbol) [0x0xbcbe6c]
	(No symbol) [0x0xba9bf6]
	(No symbol) [0x0xb7b38e]
	(No symbol) [0x0xb7c274]
	GetHandleVerifier [0x0xf9eda3+2697763]
	GetHandleVerifier [0x0xf99ec7+2677575]
	GetHandleVerifier [0x0xd44194+228884]
	GetHandleVerifier [0x0xd349f8+165496]
	GetHandleVerifier [0x0xd3b18d+192013]
	GetHandleVerifier [0x0xd247d8+99416]
	GetHandleVerifier [0x0xd24972+99826]
	GetHandleVerifier [0x0xd0ebea+10346]
	BaseThreadInitThunk [0x0x766a5d49+25]
	RtlInitializeExceptionChain [0x0x77a1d6db+107]
	RtlGetAppContainerNamedObjectPath [0x0x77a1d661+561]


In [8]:
# %% [markdown]
# 저장한 쿠키를 로드해 로그인 상태로 키워드 페이지를 열고,
# 화면에 보이는 상품 카드 정보를 수집해 CSV로 저장합니다.
# 확장프로그램 설치/설정 시간으로 60초 대기 추가.

# %%
import re, time, pickle, os, pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

COOKIE_PATH = "itemscout_cookies.pkl"
TARGET_URL = "https://itemscout.io/keyword/244728814"
OUT_CSV = "itemscout_keyword_244728814.csv"

# 대기 시간(초) — 확장프로그램 설치/설정용
WAIT_FOR_EXTENSION_SEC = 60

opts = Options()
opts.add_argument("--start-maximized")
opts.add_argument("window-size=1920x1080")
opts.add_argument("lang=ko_KR")
# 매 실행마다 확장프로그램 재설치를 피하려면 사용자 프로필을 고정:
# USER_PROFILE_DIR = os.path.abspath("./chrome_profile")
# os.makedirs(USER_PROFILE_DIR, exist_ok=True)
# opts.add_argument(f"--user-data-dir={USER_PROFILE_DIR}")
# opts.add_argument("--user-agent=Mozilla/5.0")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

def load_cookies_to_driver(driver, cookie_path, domain_hint=".itemscout.io"):
    if not os.path.exists(cookie_path):
        raise FileNotFoundError("쿠키 파일이 없습니다. 먼저 '쿠키 저장' 셀을 실행하세요.")
    # 도메인 컨텍스트 필요
    driver.get("https://itemscout.io/")
    time.sleep(2)
    with open(cookie_path, "rb") as f:
        cookies = pickle.load(f)
    for ck in cookies:
        c = ck.copy()
        c.pop("sameSite", None)
        if "domain" in c and "itemscout.io" not in c["domain"]:
            c["domain"] = domain_hint
        if "expiry" in c and not isinstance(c["expiry"], int):
            try: c["expiry"] = int(c["expiry"])
            except: c.pop("expiry", None)
        try:
            driver.add_cookie(c)
        except Exception:
            pass
    driver.refresh()
    time.sleep(2)

def is_ad(card) -> bool:
    try:
        if any(el.is_displayed() for el in card.find_elements(By.XPATH, ".//*[contains(text(),'광고')]")):
            return True
    except:
        pass
    try:
        if "광고" in card.text:
            return True
    except:
        pass
    return False

def get_first_text(el, xpaths):
    for xp in xpaths:
        try:
            t = el.find_element(By.XPATH, xp).text.strip()
            if t: return t
        except:
            continue
    return ""

def extract_with_regex(text, patterns, cast=str):
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            val = m.group(1).replace(",", "")
            try:
                return cast(val)
            except:
                return val
    return None

try:
    # ✅ 확장프로그램 설치/설정 대기
    print(f"확장프로그램 설치/설정을 진행하세요. {WAIT_FOR_EXTENSION_SEC}초 대기...")
    time.sleep(WAIT_FOR_EXTENSION_SEC)

    # 1) 쿠키 로드
    load_cookies_to_driver(driver, COOKIE_PATH)

    # 2) 타깃 페이지 열기
    driver.get(TARGET_URL)
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.XPATH, "//main//li"))
    )
    time.sleep(1)

    # 3) 카드 수집
    cards = driver.find_elements(By.XPATH, "//main//li[.//text()[contains(.,'원')] or .//*[contains(text(),'리뷰')]]")
    rows, seen = [], set()
    rank = 0

    for card in cards:
        if not card.is_displayed(): 
            continue
        if is_ad(card):
            continue

        full = card.text
        title = get_first_text(card, [
            ".//div[contains(@class,'font-bold')][1]",
            ".//p[contains(@class,'font-bold')][1]",
            ".//div[1]"
        ]) or (full.splitlines()[0].strip() if full.splitlines() else "")

        price = extract_with_regex(full, [r"([0-9,]+)\s*원"], int)
        review_cnt = extract_with_regex(full, [r"리뷰\s*([0-9,]+)"], int)
        rating = extract_with_regex(full, [r"평점\s*([0-9.]+)"], float)
        reg_date = extract_with_regex(full, [r"등록일\s*([0-9]{4}\.[0-9]{2}\.[0-9]{2})"], str)
        expected7 = extract_with_regex(full, [r"예상판매량\(?7일\)?\s*([0-9,]+)"], int)
        zzim = extract_with_regex(full, [r"찜\s*([0-9,]+)"], int)

        key = (title, price)
        if key in seen:
            continue
        seen.add(key)

        rank += 1
        rows.append({
            "rank": rank,
            "title": title,
            "price": price,
            "review_count": review_cnt,
            "rating": rating,
            "reg_date": reg_date,
            "expected_sales_7d": expected7,
            "zzim": zzim,
        })

    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"완료: {OUT_CSV} (행 {len(df)})")

finally:
    driver.quit()



확장프로그램 설치/설정을 진행하세요. 60초 대기...


InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=140.0.7339.128); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	GetHandleVerifier [0x0xd1c333+65459]
	GetHandleVerifier [0x0xd1c374+65524]
	(No symbol) [0x0xb3d973]
	(No symbol) [0x0xb2cdf0]
	(No symbol) [0x0xb4b4af]
	(No symbol) [0x0xbb0775]
	(No symbol) [0x0xbcaef9]
	(No symbol) [0x0xba9bf6]
	(No symbol) [0x0xb7b38e]
	(No symbol) [0x0xb7c274]
	GetHandleVerifier [0x0xf9eda3+2697763]
	GetHandleVerifier [0x0xf99ec7+2677575]
	GetHandleVerifier [0x0xd44194+228884]
	GetHandleVerifier [0x0xd349f8+165496]
	GetHandleVerifier [0x0xd3b18d+192013]
	GetHandleVerifier [0x0xd247d8+99416]
	GetHandleVerifier [0x0xd24972+99826]
	GetHandleVerifier [0x0xd0ebea+10346]
	BaseThreadInitThunk [0x0x766a5d49+25]
	RtlInitia